<a href="https://colab.research.google.com/github/Tomas-Turner/Unit2_Flight_Team10/blob/individual/Unit2_Tomas_BQML_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Unit 2 — Team Classification (Flights, BQML)

**Goal (team):** Build an *ops-ready* classifier in **BigQuery ML** to predict **`diverted`** on U.S. flights. Minimal handholding by design.

**What you deliver (inside this notebook):**
- One **LOGISTIC_REG** model (baseline), one **engineered** model using `TRANSFORM`
- **Evaluation** via `ML.EVALUATE` and **confusion matrices** (default 0.5 + your custom threshold)
- **Threshold choice** + 3–5 sentence ops justification
- Embedded **rubric** below (self-check before submission)

> Choose *one* dataset table that exists at your institution:  
> • `bigquery-public-data.faa.us_flights` **or** `bigquery-public-data.flights.*`  
> Make sure the table has `carrier`, `dep_delay`, `arr_delay` (for filters), `origin`, `dest`, `diverted` (or equivalent).


In [67]:
# --- Minimal setup (edit 3 vars) ---
from google.colab import auth
auth.authenticate_user()

import os
from google.cloud import bigquery

PROJECT_ID = "mgmt-467-35946"      # e.g., mgmt-467-47888
REGION     = "us-central1"
TABLE_PATH = "mgmt-467-35946.flights_data.flights_cleaned_enriched"   # or your `bigquery-public-data.flights` table/view

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"]     = REGION
bq = bigquery.Client(project=PROJECT_ID)

print("BQ Project:", PROJECT_ID)
print("Source table:", TABLE_PATH)

BQ Project: mgmt-467-35946
Source table: mgmt-467-35946.flights_data.flights_cleaned_enriched


### Quick sanity check

In [68]:

preview_sql = f"SELECT * FROM `{TABLE_PATH}` LIMIT 5"
bq.query(preview_sql).result().to_dataframe()


,FL_DATE,CARRIER,Origin,Dest,DepDelay,ArrDelay,Distance,DAY_OF_WEEK,is_arrival_delayed,Diverted
0,2024-01-07,9E,ABE,30397,1231,1450,126.0,7,True,0.0
1,2024-01-08,OH,ABE,31057,1202,1400,99.0,1,True,0.0
2,2024-01-11,9E,ABE,30397,1734,1959,111.0,4,True,0.0
3,2024-01-11,9E,ABE,30397,1235,1454,117.0,4,True,0.0
4,2024-01-11,9E,ABE,30397,1734,1959,111.0,4,True,0.0



## 1) Canonical mapping (adjust as needed)
Map to a minimal schema used in the rest of the notebook:
- `flight_date` (DATE), `dep_delay` (NUM), `distance` (NUM), `carrier` (STRING), `origin` (STRING), `dest` (STRING), `diverted` (BOOL)


In [69]:

# Adjust ONLY if your table uses different column names.
CANONICAL_BASE_SQL = f'''
WITH canonical_flights AS (
  SELECT
    CAST(COALESCE(FL_DATE, date) AS DATE) AS flight_date,
    CAST(DepDelay AS FLOAT64) AS dep_delay,
    CAST(Distance  AS FLOAT64) AS distance,
    CAST(CARRIER   AS STRING)  AS carrier,
    CAST(Origin    AS STRING)  AS origin,
    CAST(COALESCE(Dest, destination) AS STRING) AS dest,
    CAST((CASE WHEN SAFE_CAST(Diverted AS INT64)=1 OR LOWER(CAST(Diverted AS STRING))='true' THEN TRUE ELSE FALSE END) AS BOOL) AS diverted
  FROM `{TABLE_PATH}`
  WHERE DepDelay IS NOT NULL
)
'''
print(CANONICAL_BASE_SQL[:600] + "\n...")



WITH canonical_flights AS (
  SELECT
    CAST(COALESCE(FL_DATE, date) AS DATE) AS flight_date,
    CAST(DepDelay AS FLOAT64) AS dep_delay,
    CAST(Distance  AS FLOAT64) AS distance,
    CAST(CARRIER   AS STRING)  AS carrier,
    CAST(Origin    AS STRING)  AS origin,
    CAST(COALESCE(Dest, destination) AS STRING) AS dest,
    CAST((CASE WHEN SAFE_CAST(Diverted AS INT64)=1 OR LOWER(CAST(Diverted AS STRING))='true' THEN TRUE ELSE FALSE END) AS BOOL) AS diverted
  FROM `mgmt-467-35946.flights_data.flights_cleaned_enriched`
  WHERE DepDelay IS NOT NULL
)

...


### 2) Split (80/20)

In [70]:

SPLIT_CLAUSE = r'''
, split AS (
  SELECT cf.*,
         CASE WHEN RAND(12345) < 0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split
  FROM canonical_flights cf
)
'''
print(SPLIT_CLAUSE)



, split AS (
  SELECT cf.*,
         CASE WHEN RAND(12345) < 0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split
  FROM canonical_flights cf
)




## 3) Baseline model — LOGISTIC_REG (`diverted`)
Use **only** a small set of signals for the baseline (keep it honest).


In [71]:

MODEL_BASE = f"{PROJECT_ID}.unit2_flights.clf_diverted_base"

sql_baseline = f'''
{CANONICAL_BASE_SQL}
{SPLIT_CLAUSE}

CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.unit2_flights`;

CREATE OR REPLACE MODEL `{MODEL_BASE}`
OPTIONS (MODEL_TYPE='LOGISTIC_REG', INPUT_LABEL_COLS=['diverted']) AS
SELECT
  diverted,
  dep_delay, distance, carrier, origin, dest,
  EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
FROM split
WHERE split='TRAIN'
;

SELECT * FROM ML.EVALUATE(
  MODEL `{MODEL_BASE}`,
  (SELECT
     diverted,
     dep_delay, distance, carrier, origin, dest,
     EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
   FROM split WHERE split='EVAL')
);
'''
job = bq.query(sql_baseline); _ = job.result()
print("Baseline model trained:", MODEL_BASE)


BadRequest: 400 Syntax error: Unexpected keyword CREATE at [24:1]; reason: invalidQuery, location: query, message: Syntax error: Unexpected keyword CREATE at [24:1]

Location: US
Job ID: 444eab45-c7fa-497e-807d-e6608e9c362b


In [72]:
sql_create_model = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.unit2_flights.clf_diverted_base`
OPTIONS(MODEL_TYPE='LOGISTIC_REG', INPUT_LABEL_COLS=['diverted']) AS
WITH canonical_flights AS (
  SELECT
    CAST(FL_DATE AS DATE) AS flight_date,
    CAST(DepDelay AS FLOAT64) AS dep_delay,
    CAST(Distance AS FLOAT64) AS distance,
    CAST(CARRIER AS STRING) AS carrier,
    CAST(Origin AS STRING) AS origin,
    CAST(Dest AS STRING) AS dest,
    CAST(
      CASE
        WHEN SAFE_CAST(Diverted AS INT64) = 1
             OR LOWER(CAST(Diverted AS STRING)) = 'true'
             OR SAFE_CAST(Diverted AS FLOAT64) > 0
        THEN TRUE
        ELSE FALSE
      END
    AS BOOL) AS diverted
  FROM `{PROJECT_ID}.flights_data.flights_cleaned_enriched`
  WHERE DepDelay IS NOT NULL
),
split_data AS (
  SELECT *, CASE WHEN RAND() < 0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split_flag
  FROM canonical_flights
)
SELECT
  diverted,
  dep_delay,
  distance,
  carrier,
  origin,
  dest,
  EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
FROM split_data
WHERE split_flag = 'TRAIN';
"""

job = bq.query(sql_create_model)
job.result()
print("✅ Model trained successfully!")


✅ Model trained successfully!


### Confusion matrix — default 0.5 threshold

In [80]:
cm_default_sql = f"""
WITH canonical_flights AS (
  SELECT
    CAST(FL_DATE AS DATE) AS flight_date,
    CAST(DepDelay AS FLOAT64) AS dep_delay,
    CAST(Distance AS FLOAT64) AS distance,
    CAST(CARRIER AS STRING) AS carrier,
    CAST(Origin AS STRING) AS origin,
    CAST(Dest AS STRING) AS dest,
    CAST(
      CASE
        WHEN SAFE_CAST(Diverted AS INT64) = 1
             OR LOWER(CAST(Diverted AS STRING)) = 'true'
             OR SAFE_CAST(Diverted AS FLOAT64) > 0
        THEN TRUE ELSE FALSE END
    AS BOOL) AS diverted
  FROM `{PROJECT_ID}.flights_data.flights_cleaned_enriched`
  WHERE DepDelay IS NOT NULL
  LIMIT 10000
),

split AS (
  SELECT *, CASE WHEN RAND() < 0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split_flag
  FROM canonical_flights
),

scored AS (
  SELECT
    cf.diverted AS label,
    p.predicted_diverted AS pred_label,
    p.predicted_diverted_probs[OFFSET(0)].prob AS score
  FROM split AS cf
  JOIN ML.PREDICT(
    MODEL `{PROJECT_ID}.unit2_flights.clf_diverted_base`,
    (
      SELECT dep_delay, distance, carrier, origin, dest,
             EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
      FROM split
      WHERE split_flag = 'EVAL'
    )
  ) AS p
  ON TRUE
)

SELECT
  SUM(CASE WHEN label = TRUE  AND pred_label = TRUE  THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN label = FALSE AND pred_label = TRUE  THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN label = TRUE  AND pred_label = FALSE THEN 1 ELSE 0 END) AS FN,
  SUM(CASE WHEN label = FALSE AND pred_label = FALSE THEN 1 ELSE 0 END) AS TN
FROM scored;
"""
results = bq.query(cm_default_sql).result().to_dataframe()
display(results)



,TP,FP,FN,TN
0,0,0,218881,19151119


### Confusion matrix — your custom threshold

In [97]:
CUSTOM_THRESHOLD = 0.7

cm_thresh_sql = f"""
WITH canonical_flights AS (
  SELECT
    CAST(FL_DATE AS DATE) AS flight_date,
    CAST(DepDelay AS FLOAT64) AS dep_delay,
    CAST(Distance AS FLOAT64) AS distance,
    CAST(CARRIER AS STRING) AS carrier,
    CAST(Origin AS STRING) AS origin,
    CAST(Dest AS STRING) AS dest,
    CAST(
      CASE
        WHEN SAFE_CAST(Diverted AS INT64) = 1
             OR LOWER(CAST(Diverted AS STRING)) = 'true'
             OR SAFE_CAST(Diverted AS FLOAT64) > 0
        THEN TRUE ELSE FALSE END
    AS BOOL) AS diverted
  FROM `{PROJECT_ID}.flights_data.flights_cleaned_enriched`
  WHERE DepDelay IS NOT NULL
  LIMIT 10000
),

split AS (
  SELECT *, CASE WHEN RAND() < 0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split_flag
  FROM canonical_flights
),

scored AS (
  SELECT
    cf.diverted AS label,
    p.predicted_diverted_probs[OFFSET(0)].prob AS score,
    CAST(p.predicted_diverted_probs[OFFSET(0)].prob >= {CUSTOM_THRESHOLD} AS BOOL) AS pred_label
  FROM split AS cf
  JOIN ML.PREDICT(
    MODEL `{PROJECT_ID}.unit2_flights.clf_diverted_base`,
    (
      SELECT dep_delay, distance, carrier, origin, dest,
             EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
      FROM split
      WHERE split_flag = 'EVAL'
    )
  ) AS p
  ON TRUE
)

SELECT
  SUM(CASE WHEN label = TRUE  AND pred_label = TRUE  THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN label = FALSE AND pred_label = TRUE  THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN label = TRUE  AND pred_label = FALSE THEN 1 ELSE 0 END) AS FN,
  SUM(CASE WHEN label = FALSE AND pred_label = FALSE THEN 1 ELSE 0 END) AS TN
FROM scored;
"""

results = bq.query(cm_thresh_sql).result().to_dataframe()
display(results)


,TP,FP,FN,TN
0,0,0,222836,19497164



## 4) Engineered model — `TRANSFORM` (same label, stricter bar)
Create **route**, extract **day_of_week**, and **bucketize dep_delay**. Compare metrics to baseline.


In [98]:
MODEL_XFORM = f"{PROJECT_ID}.unit2_flights.clf_diverted_xform"

sql_create_model_xform = f"""
CREATE OR REPLACE MODEL `{MODEL_XFORM}`
TRANSFORM (
  CONCAT(origin, '-', dest) AS route,
  EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week,
  CASE
    WHEN dep_delay < -5  THEN 'early'
    WHEN dep_delay <=  5 THEN 'on_time'
    WHEN dep_delay <= 15 THEN 'minor'
    WHEN dep_delay <= 45 THEN 'moderate'
    ELSE 'major'
  END AS dep_delay_bucket,
  dep_delay, distance, carrier, origin, dest,
  diverted  -- ✅ include the label column explicitly
)
OPTIONS (MODEL_TYPE='LOGISTIC_REG', INPUT_LABEL_COLS=['diverted']) AS
WITH canonical_flights AS (
  SELECT
    CAST(FL_DATE AS DATE) AS flight_date,
    CAST(DepDelay AS FLOAT64) AS dep_delay,
    CAST(Distance AS FLOAT64) AS distance,
    CAST(CARRIER AS STRING) AS carrier,
    CAST(Origin AS STRING) AS origin,
    CAST(Dest AS STRING) AS dest,
    CAST(
      CASE
        WHEN SAFE_CAST(Diverted AS INT64) = 1
             OR LOWER(CAST(Diverted AS STRING)) = 'true'
             OR SAFE_CAST(Diverted AS FLOAT64) > 0
        THEN TRUE ELSE FALSE END
    AS BOOL) AS diverted
  FROM `{PROJECT_ID}.flights_data.flights_cleaned_enriched`
  WHERE DepDelay IS NOT NULL
  LIMIT 10000
),

split AS (
  SELECT *,
         CASE WHEN RAND() < 0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split_flag
  FROM canonical_flights
)

SELECT * FROM split WHERE split_flag = 'TRAIN';
"""

job = bq.query(sql_create_model_xform)
job.result()
print("✅ Engineered model trained successfully:", MODEL_XFORM)


✅ Engineered model trained successfully: mgmt-467-35946.unit2_flights.clf_diverted_xform


In [88]:
sql_compare_models = f"""
WITH canonical_flights AS (
  SELECT
    CAST(FL_DATE AS DATE) AS flight_date,
    CAST(DepDelay AS FLOAT64) AS dep_delay,
    CAST(Distance AS FLOAT64) AS distance,
    CAST(CARRIER AS STRING) AS carrier,
    CAST(Origin AS STRING) AS origin,
    CAST(Dest AS STRING) AS dest,
    CAST(
      CASE
        WHEN SAFE_CAST(Diverted AS INT64) = 1
             OR LOWER(CAST(Diverted AS STRING)) = 'true'
             OR SAFE_CAST(Diverted AS FLOAT64) > 0
        THEN TRUE ELSE FALSE END
    AS BOOL) AS diverted
  FROM `{PROJECT_ID}.flights_data.flights_cleaned_enriched`
  WHERE DepDelay IS NOT NULL
  LIMIT 10000
),

split AS (
  SELECT *,
         CASE WHEN RAND() < 0.8 THEN 'TRAIN' ELSE 'EVAL' END AS split_flag
  FROM canonical_flights
)

SELECT 'baseline' AS model_version, *
FROM ML.EVALUATE(
  MODEL `{PROJECT_ID}.unit2_flights.clf_diverted_base`,
  (
    SELECT diverted, dep_delay, distance, carrier, origin, dest,
           EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week
    FROM split WHERE split_flag = 'EVAL'
  )
)

UNION ALL

SELECT 'engineered' AS model_version, *
FROM ML.EVALUATE(
  MODEL `{MODEL_XFORM}`,
  (
    SELECT * FROM split WHERE split_flag = 'EVAL'
  )
);
"""

results = bq.query(sql_compare_models).result().to_dataframe()
display(results)


,model_version,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,baseline,0.0,0.0,0.990403,0.0,0.051119,0.720556
1,engineered,0.0,0.0,0.990802,0.0,0.052271,0.654804


In [92]:
sql_threshold_eval = f"""
WITH eval_data AS (
  SELECT
    CAST(FL_DATE AS DATE) AS flight_date,
    CAST(DepDelay AS FLOAT64) AS dep_delay,
    CAST(Distance AS FLOAT64) AS distance,
    CAST(CARRIER AS STRING) AS carrier,
    CAST(Origin AS STRING) AS origin,
    CAST(Dest AS STRING) AS dest,
    CAST(
      CASE
        WHEN SAFE_CAST(Diverted AS INT64) = 1
             OR LOWER(CAST(Diverted AS STRING)) = 'true'
             OR SAFE_CAST(Diverted AS FLOAT64) > 0
        THEN TRUE ELSE FALSE END
    AS BOOL) AS diverted
  FROM `{PROJECT_ID}.flights_data.flights_cleaned_enriched`
  WHERE DepDelay IS NOT NULL
  LIMIT 10000
),

preds AS (
  SELECT
    e.diverted AS actual_label,
    p.predicted_diverted_probs[OFFSET(0)].prob AS score,
    CASE WHEN p.predicted_diverted_probs[OFFSET(0)].prob >= 0.2 THEN TRUE ELSE FALSE END AS pred_label
  FROM ML.PREDICT(
    MODEL `{MODEL_XFORM}`,
    (SELECT dep_delay, distance, carrier, origin, dest, flight_date FROM eval_data)
  ) AS p
  JOIN eval_data AS e
  ON TRUE
)

SELECT
  SUM(CASE WHEN actual_label = TRUE  AND pred_label = TRUE  THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN actual_label = FALSE AND pred_label = TRUE  THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN actual_label = TRUE  AND pred_label = FALSE THEN 1 ELSE 0 END) AS FN,
  SUM(CASE WHEN actual_label = FALSE AND pred_label = FALSE THEN 1 ELSE 0 END) AS TN
FROM preds;
"""


In [93]:
results = bq.query(sql_threshold_eval).result().to_dataframe()
display(results)

,TP,FP,FN,TN
0,2034,177966,1127966,98692034


In [94]:
TP, FP, FN, TN = results.iloc[0]

precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Precision: 0.0113
Recall: 0.0018
F1 Score: 0.0031



### Write-up (concise)
- **Threshold chosen & ops rationale:** The threshold that I chose was 0.7, meaning that there was a stricter threshold than the original 0.5. I chose to have a higher threshold because specifically in airline operations, having a false alarm, in this case predicting a diversion when there is none, can trigger a lot of unecessary re-routing, or gate changing, or fuel planning, which are all very expensive, to I decided to have a higher threshold so that there is less false positives.
- **Baseline vs engineered — observed changes in AUC/precision/recall:Both the baseline and engineered model seem to have a 0.00 precision and recall, indicating that they almost never called for positives, either true or false. I think that this is due to the fact that the dataset had a small number of actually diverted flights, so there was not enough to show up for the recall and precision. Another difference that you can see is that the enigneered model has a slightly higher accuracy, however, when you look at the roc_auc, and the log_loss, you can see that the model actually got worse, I am not sure how that happened.
- **Risk framing:** cost of FP vs FN for diversion planning; what is your acceptable FN-rate? The cost of a False Positive is less than that of a False Negative, for a false negative you would have to plan the diversion with much less anticipation, which would be more costly, but for a false positive, although you plan everything, there is no execution of the refuling, or landing the plane, or anything like that, so you save money there. I think that a relasitc FN-rate would be about 15%.


---

## Rubric (Flights, 100 pts)
**Team-only deliverable in this notebook**

- Baseline LOGISTIC_REG + evaluation (AUC + confusion @0.5) — **20**  
- Custom threshold confusion matrix + ops justification — **20**  
- Engineered model with `TRANSFORM` (route, DOW, delay bucket) — **20**  
- Comparison table (baseline vs engineered) + 3–5 sentence interpretation — **20**  
- Reproducibility: parameters clear, no hidden magic; schema mapping documented — **10**  
- Governance notes: assumptions/limitations + slices you would monitor — **10**

> **Strictness:** No screenshots; use actual results cells. Keep explanations concise (bullet points OK).
